
Siguiendo con el ejemplo anterior, el equipo de marketing está que hecha humo. Resulta que las tiendas han empezado un nuevo programa de fidelización y que ahora tendrán usuarios, con sus nombres y sus telefonos! Podrían hacer campañas de pushes, pero primero te toca a ti, ¿como minimizar el acceso a estos datos pero permitir la operativa? 


Createmos un secret scope, con una clave de cifrado llamda aes_key.


In [0]:
# Creamos el scope https://dbc-<id>.cloud.databricks.com/#secrets/createScope
# Ponemos el secret

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.secrets.put_secret("my_security_scope","aes_key",string_value ="0123456789abcdef0123456789abcdef")

In [0]:
%sql 
CREATE TABLE IF NOT EXISTS workspace.gold.users (
  user_id long GENERATED ALWAYS AS IDENTITY,
  name STRING,    
  surname STRING, 
  phone STRING,
  email STRING,
  is_active BOOLEAN
);

INSERT INTO workspace.gold.users (name, surname, phone, email, is_active)
SELECT 
  aes_encrypt(name_plain, secret('my_security_scope', 'aes_key')) AS name,
  aes_encrypt(surname_plain, secret('my_security_scope', 'aes_key')) AS surname,
  phone,
  email,
  is_active
FROM (
  SELECT 'Carlos' AS name_plain, 'Gómez' AS surname_plain, '+34600111222' AS phone, 'carlos.gomez@example.com' AS email, true AS is_active UNION ALL
  SELECT 'Lucía', 'Martín', '+34600222333', 'lucia.martin@example.com', true UNION ALL
  SELECT 'Alejandro', 'Fernández', '+34600333444', 'alejandro.f@example.com', false UNION ALL
  SELECT 'María', 'López', '+34600444555', 'maria.lopez@example.com', true UNION ALL
  SELECT 'David', 'García', '+34600555666', 'david.garcia@example.com', true UNION ALL
  SELECT 'Paula', 'Pérez', '+34600666777', 'paula.perez@example.com', false UNION ALL
  SELECT 'Javier', 'Sánchez', '+34600777888', 'javier.sanchez@example.com', true UNION ALL
  SELECT 'Elena', 'Romero', '+34600888999', 'elena.romero@example.com', true UNION ALL
  SELECT 'Sergio', 'Torres', '+34601111222', 'sergio.torres@example.com', false UNION ALL
  SELECT 'Marta', 'Navarro', '+34601222333', 'marta.navarro@example.com', true UNION ALL
  SELECT 'Daniel', 'Ruiz', '+34601333444', 'daniel.ruiz@example.com', true UNION ALL
  SELECT 'Laura', 'Díaz', '+34601444555', 'laura.diaz@example.com', true UNION ALL
  SELECT 'Pablo', 'Serrano', '+34601555666', 'pablo.serrano@example.com', false UNION ALL
  SELECT 'Carmen', 'Muñoz', '+34601666777', 'carmen.munoz@example.com', true UNION ALL
  SELECT 'Adrian', 'Blanco', '+34601777888', 'adrian.blanco@example.com', true UNION ALL
  SELECT 'Alba', 'Molina', '+34601888999', 'alba.molina@example.com', false UNION ALL
  SELECT 'Raúl', 'Morales', '+34602111222', 'raul.morales@example.com', true UNION ALL
  SELECT 'Sara', 'Ortega', '+34602223333', 'sara.ortega@example.com', true UNION ALL
  SELECT 'Diego', 'Delgado', '+34602333444', 'diego.delgado@example.com', true UNION ALL
  SELECT 'Sofia', 'Castro', '+34602444555', 'sofia.castro@example.com', false
);

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.gold.users_decrypted AS
SELECT 
  user_id,
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('marketing') THEN 
      CAST(aes_decrypt(name, secret('my_security_scope', 'aes_key')) AS STRING)
    ELSE '***ENCRYPTED***'
  END AS name,
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('marketing') THEN 
      CAST(aes_decrypt(surname, secret('my_security_scope', 'aes_key')) AS STRING)
    ELSE '***ENCRYPTED***'
  END AS surname,
  phone,
  email,
  is_active
FROM workspace.gold.users;

In [0]:
%sql
-- Test the decryption view
SELECT * FROM workspace.gold.users_decrypted LIMIT 5;

In [0]:
%sql
select * from 